# 02 - Carga de Datos desde Git y Mapa Geografico

**Proyecto:** Indice de Ingresos Operacionales - Cali
**Equipo:** ITT Cali Inteligente - Gobierno de Datos

## Objetivo

1. Clonar/actualizar el repositorio desde GitHub
2. Cargar el archivo Excel del Registro Mercantil 2025
3. Cargar un archivo GeoJSON de comunas/zonas de Cali
4. Generar un mapa coropletico con indicadores por territorio

## 1. Instalacion de dependencias

In [ ]:
# Ejecutar solo si faltan paquetes
# !pip install pandas openpyxl geopandas folium mapclassify matplotlib

## 2. Deteccion de entorno y carga del repositorio

In [ ]:
import os
from pathlib import Path

# Configuracion del repositorio
REPO_URL = "https://github.com/j0rg3c45/Indice_ingresos_operacionales.git"
REPO_NAME = "Indice_ingresos_operacionales"

# Detectar entorno: Colab o Local
EN_COLAB = os.path.exists("/content")

if EN_COLAB:
    # En Google Colab: clonar el repo
    WORK_DIR = Path("/content") / REPO_NAME
    if not WORK_DIR.exists():
        print(f"Clonando repositorio: {REPO_URL}")
        os.system(f"git clone {REPO_URL}")
    else:
        print("Repositorio ya existe, actualizando...")
        os.system(f"cd {WORK_DIR} && git pull")
else:
    # En local: usar la carpeta del proyecto
    WORK_DIR = Path(os.getcwd()).parent
    if not (WORK_DIR / "README.md").exists():
        WORK_DIR = Path(os.getcwd())

DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Entorno: {'Google Colab' if EN_COLAB else 'Local'}")
print(f"Directorio de trabajo: {WORK_DIR}")
print(f"Directorio de datos: {DATA_DIR}")

## 3. Carga del Registro Mercantil

In [ ]:
import pandas as pd
import numpy as np

# Buscar el archivo Excel en data/
archivos_excel = list(DATA_DIR.glob("*.xlsx"))
if archivos_excel:
    ARCHIVO_EXCEL = archivos_excel[0]
    print(f"Archivo encontrado: {ARCHIVO_EXCEL.name}")
else:
    # Si no esta en el repo (por .gitignore), indicar ruta manual
    print("[!] No se encontro archivo .xlsx en data/")
    print("    Sube el archivo manualmente o ajusta la ruta:")
    ARCHIVO_EXCEL = DATA_DIR / "Registro mercantil 2025_.xlsx"

# Cargar datos
df = pd.read_excel(ARCHIVO_EXCEL, engine="openpyxl")
df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)

print(f"\nRegistros: {len(df):,}")
print(f"Columnas: {len(df.columns)}")
df.head(3)

## 4. Preparacion de indicadores por comuna

In [ ]:
# Identificar columnas clave
col_comuna = [c for c in df.columns if "comuna" in c][0]
col_ingresos = [c for c in df.columns if "ingreso" in c][0]
col_empleo = [c for c in df.columns if "personal" in c or "empleo" in c][0]
col_tamano = [c for c in df.columns if "tama" in c][0]

# Convertir ingresos a numerico
df[col_ingresos] = pd.to_numeric(df[col_ingresos], errors="coerce")
df[col_empleo] = pd.to_numeric(df[col_empleo], errors="coerce")

# Calcular indicadores por comuna
indicadores_comuna = df.groupby(col_comuna).agg(
    total_empresas=(col_comuna, "size"),
    ingresos_promedio=(col_ingresos, "mean"),
    ingresos_mediana=(col_ingresos, "median"),
    ingresos_total=(col_ingresos, "sum"),
    empleo_total=(col_empleo, "sum"),
    empleo_promedio=(col_empleo, "mean"),
).reset_index()

# Tasa de microempresas
micro = df[df[col_tamano].str.contains("MICRO", case=False, na=False)]
tasa_micro = micro.groupby(col_comuna).size().reset_index(name="n_micro")
indicadores_comuna = indicadores_comuna.merge(tasa_micro, on=col_comuna, how="left")
indicadores_comuna["pct_micro"] = (indicadores_comuna["n_micro"] / indicadores_comuna["total_empresas"] * 100).round(1)

# Diversidad economica (numero de CIIU distintos)
col_ciiu = [c for c in df.columns if "ciiu" in c and "codigo" in c][0]
diversidad = df.groupby(col_comuna)[col_ciiu].nunique().reset_index(name="n_ciiu_distintos")
indicadores_comuna = indicadores_comuna.merge(diversidad, on=col_comuna, how="left")

print(f"Indicadores calculados para {len(indicadores_comuna)} comunas")
indicadores_comuna.head(10)

## 5. Carga del GeoJSON de zonas

Carga un archivo GeoJSON con los poligonos de comunas o zonas de Cali.
Puede ser un archivo local o un ZIP descargado.

In [ ]:
import geopandas as gpd
import zipfile

# ============================================================
# CONFIGURAR AQUI: ruta al GeoJSON o ZIP con el GeoJSON
# ============================================================
# Opcion 1: archivo GeoJSON directo
# GEOJSON_PATH = DATA_DIR / "comunas_cali.geojson"

# Opcion 2: archivo ZIP que contiene el GeoJSON
# ZIP_PATH = DATA_DIR / "comunas_cali.zip"

# Buscar automaticamente archivos geoespaciales en data/
geojson_files = list(DATA_DIR.glob("**/*.geojson")) + list(DATA_DIR.glob("**/*.json"))
zip_files = list(DATA_DIR.glob("**/*.zip"))

gdf_zonas = None

if geojson_files:
    GEOJSON_PATH = geojson_files[0]
    print(f"GeoJSON encontrado: {GEOJSON_PATH.name}")
    gdf_zonas = gpd.read_file(GEOJSON_PATH)

elif zip_files:
    ZIP_PATH = zip_files[0]
    print(f"ZIP encontrado: {ZIP_PATH.name}")
    # Descomprimir
    extract_dir = DATA_DIR / ZIP_PATH.stem
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(extract_dir)
    # Buscar GeoJSON dentro del ZIP extraido
    geojson_extraidos = list(extract_dir.glob("**/*.geojson")) + list(extract_dir.glob("**/*.json"))
    if geojson_extraidos:
        GEOJSON_PATH = geojson_extraidos[0]
        print(f"  -> GeoJSON extraido: {GEOJSON_PATH.name}")
        gdf_zonas = gpd.read_file(GEOJSON_PATH)
    else:
        # Intentar con shapefiles
        shp_files = list(extract_dir.glob("**/*.shp"))
        if shp_files:
            print(f"  -> Shapefile encontrado: {shp_files[0].name}")
            gdf_zonas = gpd.read_file(shp_files[0])

else:
    print("[!] No se encontro archivo GeoJSON ni ZIP en data/")
    print("    Sube un archivo .geojson o .zip con poligonos de comunas de Cali")
    print("    Ejemplo: data/comunas_cali.geojson")

if gdf_zonas is not None:
    # Normalizar CRS a WGS84
    if gdf_zonas.crs is None:
        gdf_zonas = gdf_zonas.set_crs("EPSG:4326")
    elif gdf_zonas.crs.to_epsg() != 4326:
        gdf_zonas = gdf_zonas.to_crs("EPSG:4326")
    
    print(f"\nPoligonos cargados: {len(gdf_zonas)}")
    print(f"CRS: {gdf_zonas.crs}")
    print(f"Columnas: {list(gdf_zonas.columns)}")
    gdf_zonas.head()

## 6. Cruce espacial: Indicadores + GeoJSON

In [ ]:
if gdf_zonas is not None:
    # Identificar la columna de comuna en el GeoJSON
    # Buscar columnas que contengan 'comuna', 'COMUNA', 'nombre', 'NOMBRE'
    cols_geo = gdf_zonas.columns.str.lower()
    col_join_geo = None
    for patron in ["comuna", "nombre", "name", "id"]:
        matches = [c for c, cl in zip(gdf_zonas.columns, cols_geo) if patron in cl]
        if matches:
            col_join_geo = matches[0]
            break
    
    if col_join_geo:
        print(f"Columna de join en GeoJSON: '{col_join_geo}'")
        print(f"Valores ejemplo: {gdf_zonas[col_join_geo].head(5).tolist()}")
        print(f"\nColumna de join en datos: '{col_comuna}'")
        print(f"Valores ejemplo: {indicadores_comuna[col_comuna].head(5).tolist()}")
        
        # Intentar merge directo
        gdf_merged = gdf_zonas.merge(
            indicadores_comuna,
            left_on=col_join_geo,
            right_on=col_comuna,
            how="left"
        )
        
        n_match = gdf_merged["total_empresas"].notna().sum()
        print(f"\nZonas con datos cruzados: {n_match} de {len(gdf_merged)}")
        
        if n_match == 0:
            print("\n[!] No hubo match. Revisa que los nombres de comuna coincidan.")
            print(f"    GeoJSON: {gdf_zonas[col_join_geo].unique()[:5]}")
            print(f"    Datos:   {indicadores_comuna[col_comuna].unique()[:5]}")
    else:
        print("[!] No se encontro columna de comuna en el GeoJSON.")
        print(f"    Columnas disponibles: {list(gdf_zonas.columns)}")
        gdf_merged = None
else:
    print("[!] No hay GeoJSON cargado. Sube un archivo en data/")
    gdf_merged = None

## 7. Mapa coropletico - Total de empresas por comuna

In [ ]:
import matplotlib.pyplot as plt

if gdf_merged is not None and "total_empresas" in gdf_merged.columns:
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    gdf_merged.plot(
        column="total_empresas",
        cmap="YlOrRd",
        linewidth=0.8,
        edgecolor="0.3",
        legend=True,
        legend_kwds={"label": "Total de empresas", "shrink": 0.7},
        ax=ax,
        missing_kwds={"color": "lightgrey", "label": "Sin datos"}
    )
    
    ax.set_title("Densidad Empresarial por Comuna - Cali 2025", fontsize=14, fontweight="bold")
    ax.set_axis_off()
    plt.tight_layout()
    
    # Guardar
    fig.savefig(OUTPUT_DIR / "mapa_densidad_empresarial.png", dpi=150, bbox_inches="tight")
    print("[OK] Mapa guardado: outputs/mapa_densidad_empresarial.png")
    plt.show()
else:
    print("[!] No se puede generar mapa. Verifica el cruce de datos.")

## 8. Mapa coropletico - Ingresos promedio por comuna

In [ ]:
if gdf_merged is not None and "ingresos_promedio" in gdf_merged.columns:
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    gdf_merged.plot(
        column="ingresos_promedio",
        cmap="Blues",
        linewidth=0.8,
        edgecolor="0.3",
        legend=True,
        legend_kwds={"label": "Ingresos promedio ($)", "shrink": 0.7},
        ax=ax,
        missing_kwds={"color": "lightgrey", "label": "Sin datos"}
    )
    
    ax.set_title("Ingresos Operacionales Promedio por Comuna - Cali 2025", fontsize=14, fontweight="bold")
    ax.set_axis_off()
    plt.tight_layout()
    
    fig.savefig(OUTPUT_DIR / "mapa_ingresos_promedio.png", dpi=150, bbox_inches="tight")
    print("[OK] Mapa guardado: outputs/mapa_ingresos_promedio.png")
    plt.show()
else:
    print("[!] No se puede generar mapa de ingresos.")

## 9. Mapa interactivo con Folium

In [ ]:
import folium

if gdf_merged is not None and "total_empresas" in gdf_merged.columns:
    # Centro de Cali
    centro = [3.4516, -76.5320]
    
    m = folium.Map(location=centro, zoom_start=12, tiles="CartoDB positron")
    
    # Capa coropletica
    folium.Choropleth(
        geo_data=gdf_merged.to_json(),
        data=gdf_merged,
        columns=[col_join_geo, "total_empresas"],
        key_on=f"feature.properties.{col_join_geo}",
        fill_color="YlOrRd",
        fill_opacity=0.7,
        line_opacity=0.5,
        legend_name="Total de empresas por comuna",
        name="Densidad empresarial"
    ).add_to(m)
    
    # Tooltips con info
    folium.GeoJson(
        gdf_merged.to_json(),
        name="Info por comuna",
        tooltip=folium.GeoJsonTooltip(
            fields=[col_join_geo, "total_empresas", "empleo_total", "pct_micro"],
            aliases=["Comuna", "Empresas", "Empleo total", "% Micro"],
            localize=True
        ),
        style_function=lambda x: {"fillOpacity": 0, "weight": 0}
    ).add_to(m)
    
    # Capas base adicionales
    folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
        name="Esri Satelite"
    ).add_to(m)
    
    folium.LayerControl().add_to(m)
    
    # Guardar HTML
    m.save(str(OUTPUT_DIR / "mapa_interactivo_empresas.html"))
    print("[OK] Mapa interactivo guardado: outputs/mapa_interactivo_empresas.html")
    
    m
else:
    print("[!] No se puede generar mapa interactivo. Verifica el cruce de datos.")

## 10. Tabla resumen de indicadores por comuna

In [ ]:
# Mostrar tabla ordenada por total de empresas
resumen = indicadores_comuna.sort_values("total_empresas", ascending=False).copy()
resumen["ingresos_promedio_fmt"] = resumen["ingresos_promedio"].apply(
    lambda x: f"${x/1e6:.1f}M" if pd.notna(x) else "N/A"
)
resumen["ingresos_total_fmt"] = resumen["ingresos_total"].apply(
    lambda x: f"${x/1e9:.2f}B" if pd.notna(x) and x >= 1e9 else (f"${x/1e6:.0f}M" if pd.notna(x) else "N/A")
)

cols_mostrar = [col_comuna, "total_empresas", "ingresos_promedio_fmt", "ingresos_total_fmt",
               "empleo_total", "pct_micro", "n_ciiu_distintos"]
print(resumen[cols_mostrar].to_string(index=False))

## 11. Notas

- Si el GeoJSON no hace match con los nombres de comuna del Registro Mercantil, ajustar los nombres en la celda 6.
- El archivo Excel no se versiona en git (esta en .gitignore por peso). Debe estar en `data/` localmente.
- Para agregar un nuevo GeoJSON, colocarlo en `data/` como `.geojson` o como `.zip` conteniendo el archivo.
- Los mapas se guardan en `outputs/` como PNG y HTML.